# UHVDB r6 summary counts

Counts from the released UHVDB r6 metadata table (`uhvdb_metadata.tsv.gz`).

- **Total sequences:** metadata rows
- **Unique sequences:** distinct `hash` values
- **Database representation:** proportion of unique hashes with at least one sequence whose `db_type` is Database
- **VIRE and UHGV representation:** proportion of unique hashes with at least one sequence from each `source_db`
- **Airways, skin, and urogenital assemblies:** proportion of unique hashes with at least one sequence from that body site and no Database sequence
- **Complete viruses:** proportion of unique hashes with at least one sequence whose `checkv_quality` is Complete
- **Genomovars, species, subgenera, genera, subfamilies, families:** distinct cluster IDs

In [1]:
from pathlib import Path

import polars as pl

METADATA_PATH = Path(
    "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB"
    "/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz"
)

COLS = [
    "hash",
    "source_db",
    "db_type",
    "body_site",
    "checkv_quality",
    "genomovar_cluster_id",
    "species_cluster_id",
    "subgenus_cluster_id",
    "genus_cluster_id",
    "subfamily_cluster_id",
    "family_cluster_id",
]

df = pl.read_csv(
    METADATA_PATH,
    separator="\t",
    columns=COLS,
)

print(f"Loaded {df.height:,} rows from {METADATA_PATH}")

Loaded 1,541,381 rows from /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz


In [2]:
def n_distinct(col: str) -> int:
    return df.select(pl.col(col).drop_nulls().n_unique()).item()


counts = [
    ("total sequences", df.height),
    ("unique sequences", n_distinct("hash")),
    ("genomovars", n_distinct("genomovar_cluster_id")),
    ("species", n_distinct("species_cluster_id")),
    ("subgenera", n_distinct("subgenus_cluster_id")),
    ("genera", n_distinct("genus_cluster_id")),
    ("subfamilies", n_distinct("subfamily_cluster_id")),
    ("families", n_distinct("family_cluster_id")),
]

width = max(len(label) for label, _ in counts)
for label, n in counts:
    print(f"{label:<{width}}  {n:>10,}")

total sequences    1,541,381
unique sequences   1,155,490
genomovars           925,604
species              291,605
subgenera            133,987
genera                53,506
subfamilies           16,401
families               1,590


In [3]:
n_unique_hashes = df.select(pl.col("hash").drop_nulls().n_unique()).item()

def n_hashes_with(expr: pl.Expr) -> int:
    return (
        df.filter(expr)
        .select(pl.col("hash").drop_nulls().n_unique())
        .item()
    )


for label, n in [
    ("Database", n_hashes_with(pl.col("db_type") == "Database")),
    ("VIRE", n_hashes_with(pl.col("source_db") == "VIRE")),
    ("UHGV", n_hashes_with(pl.col("source_db") == "UHGV")),
]:
    prop = n / n_unique_hashes
    print(
        f"unique hashes with at least one {label} sequence: "
        f"{prop:.1%} ({n:,} / {n_unique_hashes:,})"
    )

database_hashes = df.filter(pl.col("db_type") == "Database").select("hash").unique()
for body_site in ("Airways", "Skin", "Urogenital"):
    n_site = (
        df.filter(
            (pl.col("body_site") == body_site)
            & (~pl.col("hash").is_in(database_hashes["hash"]))
        )
        .select(pl.col("hash").drop_nulls().n_unique())
        .item()
    )
    prop_site = n_site / n_unique_hashes
    print(
        f"unique {body_site.lower()} hashes with exclusively Assembly sequences: "
        f"{prop_site:.1%} ({n_site:,} / {n_unique_hashes:,})"
    )

n_complete_hashes = n_hashes_with(pl.col("checkv_quality") == "Complete")
prop_complete = n_complete_hashes / n_unique_hashes
print(
    "unique hashes with at least one Complete sequence: "
    f"{prop_complete:.1%} ({n_complete_hashes:,} / {n_unique_hashes:,})"
)

unique hashes with at least one Database sequence: 89.4% (1,032,750 / 1,155,490)
unique hashes with at least one VIRE sequence: 37.5% (433,678 / 1,155,490)
unique hashes with at least one UHGV sequence: 15.7% (181,979 / 1,155,490)


/tmp/ipykernel_3712656/3072175054.py:25: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df.filter(


unique airways hashes with exclusively Assembly sequences: 6.4% (74,201 / 1,155,490)
unique skin hashes with exclusively Assembly sequences: 0.8% (9,023 / 1,155,490)
unique urogenital hashes with exclusively Assembly sequences: 0.2% (2,423 / 1,155,490)
unique hashes with at least one Complete sequence: 27.3% (314,905 / 1,155,490)
